# SI26-Week2-Humna
**Project:** Urdu OCR — Code Saviours ML/AI Internship (Batch SI-26)
**Week 2:** Image Preprocessing + Testing Existing OCR Tools

This notebook:
1. Loads the raw Urdu text images from `data/raw/`
2. Preprocesses them (grayscale → aspect-preserving resize/pad → denoise → binarise) and saves the results to `data/processed/`
3. Runs baseline Tesseract OCR on the processed images to see how well an *existing* OCR tool handles Urdu
4. Documents a gap analysis explaining why Tesseract struggles, motivating why this project needs a custom model

**Note on this version:** an earlier attempt at this notebook used a fixed global threshold (`cv2.threshold(img, 127, 255, ...)`) and a hard resize to `(512, 128)`. On real photographed pages with uneven lighting, a single fixed brightness cutoff turns shadowed regions into speckled noise ("dotted" output), and forcing every image into the same box regardless of its original aspect ratio stretches/warps the Urdu characters. This version fixes both: it uses **Otsu's method** (which recalculates the best threshold per image automatically) and an **aspect-ratio-preserving resize with padding**.


## Part A — Preprocess Your Images

### Step 2 — Install Libraries

In [1]:
!pip install opencv-python-headless pillow matplotlib

import cv2
import numpy as np
from PIL import Image
import os
import glob
import matplotlib.pyplot as plt

print('Libraries loaded successfully!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 42.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 41.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Libraries loaded successfully!


### Step 3 — Preprocessing Function (fixed version)

Changes from the original handout version, and why:

| Old (caused dotted output) | New | Why |
|---|---|---|
| `cv2.resize(gray, (512, 128))` — forces every image into the same box | Resize preserving aspect ratio, then pad onto a fixed white canvas | Hard resize stretches/squashes characters — bad for joined Urdu script |
| `cv2.threshold(img, 127, 255, THRESH_BINARY)` — one fixed brightness cutoff for the whole image | `cv2.threshold(img, 0, 255, THRESH_BINARY + THRESH_OTSU)` | Otsu recalculates the optimal cutoff per image, so uneven lighting/shadows don't turn into speckled dots |
| Denoise strength `h=10` | Denoise strength `h=7` | Slightly gentler — strong denoising can erode thin Urdu strokes and diacritic dots before they even reach the threshold step |


In [2]:
def preprocess_image(image_path, save_path, target_size=(800, 160)):
    """
    Preprocess a single image for OCR:
    1. Grayscale
    2. Resize preserving aspect ratio, padded onto a fixed white canvas (no stretching)
    3. Light denoising
    4. Otsu binarisation (auto threshold per image -> no speckling on uneven lighting)
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f'Could not load: {image_path}')
        return None

    # Step 1: Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 2: Resize preserving aspect ratio, then pad onto a white canvas
    target_w, target_h = target_size
    h, w = gray.shape
    scale = min(target_w / w, target_h / h)
    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
    resized = cv2.resize(gray, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

    canvas = np.full((target_h, target_w), 255, dtype=np.uint8)
    y_off = (target_h - new_h) // 2
    x_off = (target_w - new_w) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized

    # Step 3: Light denoise (too strong erodes thin strokes/dots in Urdu script)
    denoised = cv2.fastNlMeansDenoising(canvas, h=7)

    # Step 4: Otsu binarisation - auto threshold per image, fixes 'dotted' output
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    cv2.imwrite(save_path, binary)
    return binary

os.makedirs('data/processed', exist_ok=True)
print('Preprocessing function ready!')


Preprocessing function ready!


In [3]:
# Find all images in data/raw/
all_images = glob.glob('data/raw/**/*.jpg', recursive=True)
all_images += glob.glob('data/raw/**/*.jpeg', recursive=True)
all_images += glob.glob('data/raw/**/*.png', recursive=True)

print(f'Found {len(all_images)} images to process')

processed_count = 0
for img_path in all_images:
    filename = os.path.splitext(os.path.basename(img_path))[0] + '.png'
    save_path = f'data/processed/{filename}'
    result = preprocess_image(img_path, save_path)
    if result is not None:
        processed_count += 1

print(f'Done! Processed {processed_count} images')
print('Check data/processed/ folder')


Found 0 images to process
Done! Processed 0 images
Check data/processed/ folder


### Quick visual sanity check
Before moving to OCR, eyeball a few processed images to confirm the dotted-noise issue is gone (should be clean black text on white, no speckling).

In [4]:
sample_paths = glob.glob('data/processed/*.png')[:4]

fig, axes = plt.subplots(1, len(sample_paths), figsize=(16, 4))
if len(sample_paths) == 1:
    axes = [axes]
for ax, p in zip(axes, sample_paths):
    ax.imshow(plt.imread(p), cmap='gray')
    ax.set_title(os.path.basename(p), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()


ValueError: Number of columns must be a positive integer, not 0

<Figure size 1600x400 with 0 Axes>

## Part B — Test Tesseract OCR on Your Urdu Images

In [ ]:
!apt-get install -y tesseract-ocr tesseract-ocr-urd
!pip install pytesseract

import pytesseract
from PIL import Image

# Test on 5 of your processed images
test_images = list(glob.glob('data/processed/*.png'))[:5]

print('=== Tesseract Results on Urdu Images ===')
print()
for img_path in test_images:
    img = Image.open(img_path)
    # 'urd' tells Tesseract to use the Urdu language model
    result = pytesseract.image_to_string(img, lang='urd')
    print(f'Image: {img_path}')
    print(f'Tesseract output: {result}')
    print('---')


### Step 4 — Gap Analysis

For **each of the 5 test images above**, fill in the table below (replace the placeholders with what you actually observe):

| Image | Actual Urdu text | Tesseract output | What went wrong |
|---|---|---|---|
| image_1.png | *(type the real text here)* | *(paste Tesseract's output here)* | *(wrong characters / missing words / gibberish / merged letters — describe it)* |
| image_2.png | | | |
| image_3.png | | | |
| image_4.png | | | |
| image_5.png | | | |

**Summary paragraph** (start with this sentence and finish it based on your actual results):

> "Tesseract fails on Urdu because ..."

*(Think about: Urdu is written right-to-left in a cursive, context-dependent script where a letter's shape changes depending on its position in a word (Nastaliq style); Tesseract's Urdu model is trained mostly on cleaner, printed Naskh-style text; diacritics/dots are easy to lose during binarisation; word segmentation assumes left-to-right Latin-style spacing. Adjust this to match what you actually saw in your 5 images.)*

Copy this analysis into your GitHub README under a new section called **Why We Need a Better Model**.
